# M7-B1 — Mesures d audit (à compléter)

## 1. Disparate impact du modèle — puis investigation

DI sur prédictions et étiquettes, puis FNR/FPR et probabilité moyenne **par groupe** contre une référence construite depuis `dms_jours`.

In [3]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv("../data/dms_dataset.csv")
model = joblib.load("../legacy/dms_predictor_v1.joblib")

X = df[["age", "nb_comorbidites", "imc"]].assign(sexe_bin=(df["sexe"] == "M").astype(int))
df["pred"] = model.predict(X)
df["proba"] = model.predict_proba(X)[:, 1]

In [5]:
# --- 1. Disparate impact : sur les prédictions ET sur l'étiquette ---
rate_pred = df.groupby("sexe")["pred"].mean()
rate_label = df.groupby("sexe")["sejour_prolonge"].mean()

di_pred = rate_pred["F"] / rate_pred["M"]
di_label = rate_label["F"] / rate_label["M"]

print("DI F/M (prédictions) =", round(di_pred, 3))
print("DI F/M (étiquettes)  =", round(di_label, 3))
print("Repère conventionnel 4/5 (0.80) — signal d'alerte, pas un verdict.")

DI F/M (prédictions) = 0.291
DI F/M (étiquettes)  = 0.653
Repère conventionnel 4/5 (0.80) — signal d'alerte, pas un verdict.


In [11]:
# --- 2. Référence construite depuis dms_jours ---
# Seuil de dms_jours a partir duquel sejour_prolonge=1, observe par groupe (a documenter/discuter)
seuils = df.groupby("sexe").apply(
    lambda g: g.loc[g["sejour_prolonge"] == 1, "dms_jours"].min(),
    include_groups=False
)
print("\nSeuil dms_jours associe a sejour_prolonge=1, par groupe :")
print(seuils)

seuil_commun = seuils.min()  # a discuter : seuil unique retenu comme reference
df["y_ref"] = (df["dms_jours"] >= seuil_commun).astype(int)


Seuil dms_jours associe a sejour_prolonge=1, par groupe :
sexe
F    5.6
M    5.6
dtype: float64


In [9]:
# --- 3. FNR / FPR par groupe, pour l'etiquette ET pour le modele ---
def error_rates(g, col_pred):
    pos, neg = g[g["y_ref"] == 1], g[g["y_ref"] == 0]
    return pd.Series({
        "FNR": 1 - pos[col_pred].mean(),
        "FPR": neg[col_pred].mean(),
    })

print("\nFNR/FPR par groupe -- etiquette (sejour_prolonge) :")
print(df.groupby("sexe").apply(lambda g: error_rates(g, "sejour_prolonge"), include_groups=False).round(3))

print("\nFNR/FPR par groupe -- modele (pred) :")
print(df.groupby("sexe").apply(lambda g: error_rates(g, "pred"), include_groups=False).round(3))


FNR/FPR par groupe -- etiquette (sejour_prolonge) :
        FNR  FPR
sexe            
F     0.363  0.0
M     0.000  0.0

FNR/FPR par groupe -- modele (pred) :
        FNR    FPR
sexe              
F     0.737  0.018
M     0.242  0.223


In [10]:
# --- 4. Calibration par groupe : probabilite moyenne predite vs taux reel ---
calib = df.groupby("sexe").apply(lambda g: pd.Series({
    "proba_moyenne_predite": g["proba"].mean(),
    "taux_reel_y_ref": g["y_ref"].mean(),
}), include_groups=False)
print("\nCalibration par groupe (proba moyenne predite vs taux reel) :")
print(calib.round(3))


Calibration par groupe (proba moyenne predite vs taux reel) :
      proba_moyenne_predite  taux_reel_y_ref
sexe                                        
F                     0.321            0.504
M                     0.491            0.491


## 2. Ressources (psutil)

In [ ]:
# TODO


## 3. Comparaison à 2 alternatives

In [ ]:
# TODO
